In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/shl-datacard/train_cleaned.csv
/kaggle/input/shl-datacard/test_cleaned.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/sample_submission.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/train.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/test.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_885.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_1142.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_1006.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_817.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_765.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_508.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_257.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_330.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_72.wav
/kaggle/input/shl-inter

In [38]:
!pip install -q transformers datasets torchaudio

import os
import torch
import torchaudio
import numpy as np
import pandas as pd

In [40]:
# Dataset Loading


train_df = pd.read_csv("/kaggle/input/shl-datacard/train_cleaned.csv")  
AUDIO_DIR = "/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/train" 

In [41]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchaudio
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #GPU


In [42]:
#Bert Model

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased").to(device)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
#For Audio_path

train_df['audio_path'] = train_df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x))

texts = train_df['filename'].tolist()     
audio_paths = train_df['audio_path'].tolist()
labels = train_df['label'].tolist()

In [47]:


missing_files = []
for path in train_df['audio_path']:
    if not os.path.isfile(path):
        missing_files.append(path)

print(f"Missing files: {missing_files}")


Missing files: []


In [48]:
train_df.head()

,filename,label,transcript,clean_transcript,audio_path
0,audio_710.wav,1.0,sardbraadesh நீ깨 அது என்று தெல்லுது இங்கோடுத்...,sardbraadesh நீ깨 அது என்று தெல்லுது இங்கோடுத்த...,/kaggle/input/shl-intern-hiring-assessment/Dat...
1,audio_1265.wav,1.0,My favorite hobby is cultivation of plants su...,my favorite hobby is cultivation of plants suc...,/kaggle/input/shl-intern-hiring-assessment/Dat...
2,audio_1114.wav,1.5,My Girl is to become an electrical employee ....,my girl is to become an electrical employee. a...,/kaggle/input/shl-intern-hiring-assessment/Dat...
3,audio_946.wav,1.5,the playground looks like very clear and neat...,the playground looks very clear and neat as th...,/kaggle/input/shl-intern-hiring-assessment/Dat...
4,audio_1127.wav,2.0,My goal is to bring my parents to live with m...,my goal is to bring my parents to live with me...,/kaggle/input/shl-intern-hiring-assessment/Dat...


In [50]:


NUM_FOLDS = 5
TEXT_COL = "clean_transcript"
LABEL_COL = "label"
MAX_LEN = 195

# Datset Class for Bert model
class RegressionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i], dtype=torch.float),
        }

# Evaluation functions(MAE & Pearson)
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    mae = mean_absolute_error(labels, preds)
    
    pearson_corr = pearsonr(labels, preds)[0]
    return {"mae": mae, "pearson": pearson_corr}


kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

all_mae,  all_pearson = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"\n===== Fold {fold + 1}/{NUM_FOLDS} =====")

    train_texts = train_df.iloc[train_idx][TEXT_COL]
    train_labels = train_df.iloc[train_idx][LABEL_COL]
    val_texts = train_df.iloc[val_idx][TEXT_COL]
    val_labels = train_df.iloc[val_idx][LABEL_COL]

    train_ds = RegressionDataset(train_texts, train_labels, tokenizer)
    val_ds = RegressionDataset(val_texts, val_labels, tokenizer)

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=1                         # Bert Model with a regression head
    ).to(device)

    args = TrainingArguments(
        output_dir=f"./bert_regressor_fold_{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=9,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
   
    trainer.save_model(f"./saved_model_fold_{fold}")  

  
    preds_output = trainer.predict(val_ds)
    preds = preds_output.predictions.flatten()
    true = val_labels.values
    mae = mean_absolute_error(true, preds)
    pearson_corr = pearsonr(true, preds)[0]

    print(f"Fold {fold + 1} - MAE: {mae:.4f}, Pearson: {pearson_corr:.4f}")

    all_mae.append(mae)
  
    all_pearson.append(pearson_corr)

# Final aggregated metrics
print("\n==== Cross-Validation Results ====")
print(f"Avg MAE:     {np.mean(all_mae):.4f} ± {np.std(all_mae):.4f}")

print(f"Avg Pearson: {np.mean(all_pearson):.4f} ± {np.std(all_pearson):.4f}")



===== Fold 1/5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Mae,Pearson
1,No log,5.419926,2.039755,-0.035242
2,No log,1.954834,1.207574,0.039066
3,No log,1.288671,0.992122,0.037937
4,No log,1.282737,0.945125,0.432657
5,3.647100,0.908224,0.766282,0.598119
6,3.647100,0.786914,0.652926,0.650028
7,3.647100,0.700441,0.594968,0.683357
8,3.647100,0.759257,0.590244,0.685428
9,3.647100,0.775635,0.595080,0.680618


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Fold 4 - MAE: 0.6984, Pearson: 0.5740

===== Fold 5/5 =====


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Mae,Pearson
1,No log,6.039679,2.237602,-0.238884
2,No log,2.084574,1.275287,-0.212588
3,No log,1.103404,0.884176,-0.168129
4,No log,0.976330,0.824451,0.363705
5,3.644100,0.816333,0.749713,0.548291
6,3.644100,0.726247,0.639655,0.562914
7,3.644100,0.694201,0.643645,0.605367
8,3.644100,0.705462,0.635890,0.600524
9,3.644100,0.711629,0.640246,0.599987


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

Fold 5 - MAE: 0.6402, Pearson: 0.6000

==== Cross-Validation Results ====
Avg MAE:     0.6180 ± 0.0496
Avg Pearson: 0.6245 ± 0.0609


In [51]:

# BERT Embedding Extraction

def extract_bert_embeddings(model, dataset, batch_size=32):
    model.eval()
    dataloader = DataLoader(dataset, batch_size=batch_size)
    embeddings = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token

            embeddings.append(cls_embeddings.cpu())

    embeddings = torch.cat(embeddings, dim=0)
    return embeddings.numpy()


In [52]:
best_fold = 2 # Best Model Loading


best_model_path = f"./saved_model_fold_{best_fold}"
fine_tuned_model = BertForSequenceClassification.from_pretrained(best_model_path, num_labels=1).to(device)

In [83]:
test_df = pd.read_csv("/kaggle/input/shl-datacard/test_cleaned.csv")

In [93]:
test_texts = test_df[TEXT_COL].fillna("").astype(str).tolist()

In [85]:
Train_texts = train_df[TEXT_COL]
Train_labels = train_df[LABEL_COL]

Test_texts = test_df[TEXT_COL]




In [109]:
Test_texts.isnull().sum()


4

In [110]:
Test_texts = Test_texts.fillna("Missing text")


In [112]:
Test_texts.isnull().sum()

0

In [102]:
Test_texts

0      my hobbies are playing cricket because i am a ...
1      my favorite place is in andhra padesh. it is i...
2      yeah, my best days in my life is recently i go...
3      actually the most i use it to enjoy is practic...
4      i would to become a beauty unit. among the art...
                             ...                        
199    my favorite hobby is betting on nba and footba...
200    my favorite place to travel is to ocean city, ...
201    the journey to switzerland is an adventure in ...
202    , my goal in life is to live a happy fulfillin...
203    the school playground looks pretty big. there ...
Name: clean_transcript, Length: 204, dtype: object

In [68]:
train_DF=RegressionDataset(Train_texts, Train_labels, tokenizer) #Training Dataset class


In [113]:
test_texts = Test_texts.tolist()  
dummy_labels = np.zeros(len(test_texts))




test_dataset = RegressionDataset(Test_texts, dummy_labels, tokenizer) #Test Dataset Class




In [114]:
test_bert_embeds = extract_bert_embeddings(fine_tuned_model, test_dataset)  


In [126]:
test_bert_embeds.shape

(204, 768)

In [69]:
bert_embeds = extract_bert_embeddings(fine_tuned_model, train_DF )  

In [70]:
bert_embeds.shape

(424, 768)

In [54]:
!pip install transformers torchaudio


In [55]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor # For Audio Embeddings
import torchaudio


wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [56]:
#preprocessing the audio files (like sampling, trimming silence, normlaisation,etc.)


import librosa
import soundfile as sf
import os
from tqdm import tqdm

def preprocess_audio(input_path, output_path, target_sr=16000):
   
    y, sr = librosa.load(input_path, sr=None)

  
    y_resampled = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

    
    y_trimmed, _ = librosa.effects.trim(y_resampled, top_db=20)

    
    y_normalized = librosa.util.normalize(y_trimmed)

    
    sf.write(output_path, y_normalized, target_sr)


def batch_preprocess(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for file in tqdm(os.listdir(input_dir)):
        if file.endswith(".wav"):
            preprocess_audio(
                input_path=os.path.join(input_dir, file),
                output_path=os.path.join(output_dir, file)
            )


batch_preprocess("/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/train", "processed_audio/train/")
batch_preprocess("/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test", "processed_audio/test/")


100%|██████████| 204/204 [01:17<00:00,  2.62it/s]


In [71]:
#Audio Embedding Extraction

def extract_audio_embedding(audio_path, model, processor, device):
   
    waveform, sample_rate = torchaudio.load(audio_path)
    
   
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0).unsqueeze(0)

   
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000

    inputs = processor(waveform.squeeze(0), sampling_rate=sample_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(inputs.input_values.to(device))

  
    embeddings = outputs.last_hidden_state.mean(dim=1)  

    return embeddings.squeeze(0).cpu().numpy() 


In [127]:
train_audio_folder= "/kaggle/working/processed_audio/train"

audio_embeddings = []

for fname in train_df['filename']:
    audio_path = os.path.join(train_audio_folder, fname)
    emb = extract_audio_embedding(audio_path, wav2vec_model, wav2vec_processor, device)
    audio_embeddings.append(emb)


AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [76]:
audio_embeddings = np.vstack(audio_embeddings)

In [77]:
audio_embeddings.shape


(424, 768)

In [79]:
X_train = np.concatenate([bert_embeds, audio_embeddings], axis=1) #Concatenating Bert+audio embeddings for XGB.
Y_train = Train_labels.values    #Ground Truth Levels.



In [90]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Data Splitting
X_tr, X_val, y_tr, y_val = train_test_split(X_train, Y_train, test_size=0.1, random_state=42)


xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    seed=42,
    tree_method='gpu_hist',  
)


xgb_model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], early_stopping_rounds=50, verbose=True)




[0]	validation_0-rmse:1.04694	validation_1-rmse:1.02692
[1]	validation_0-rmse:1.03829	validation_1-rmse:1.01890
[2]	validation_0-rmse:1.02961	validation_1-rmse:1.01044


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [06:43:10] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


[3]	validation_0-rmse:1.02165	validation_1-rmse:1.00267
[4]	validation_0-rmse:1.01321	validation_1-rmse:0.99540
[5]	validation_0-rmse:1.00522	validation_1-rmse:0.98778
[6]	validation_0-rmse:0.99705	validation_1-rmse:0.98247
[7]	validation_0-rmse:0.98900	validation_1-rmse:0.97633
[8]	validation_0-rmse:0.98137	validation_1-rmse:0.96948
[9]	validation_0-rmse:0.97340	validation_1-rmse:0.96113
[10]	validation_0-rmse:0.96579	validation_1-rmse:0.95528
[11]	validation_0-rmse:0.95809	validation_1-rmse:0.94870
[12]	validation_0-rmse:0.95019	validation_1-rmse:0.94459
[13]	validation_0-rmse:0.94265	validation_1-rmse:0.93696
[14]	validation_0-rmse:0.93515	validation_1-rmse:0.93171
[15]	validation_0-rmse:0.92753	validation_1-rmse:0.92315
[16]	validation_0-rmse:0.92022	validation_1-rmse:0.91623
[17]	validation_0-rmse:0.91290	validation_1-rmse:0.90972
[18]	validation_0-rmse:0.90579	validation_1-rmse:0.90216
[19]	validation_0-rmse:0.89865	validation_1-rmse:0.89662
[20]	validation_0-rmse:0.89148	validat

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [99]:

for i in range(5):
    print(test_dataset[i])  

{'input_ids': tensor([  101,  2026,  7570, 27982,  2024,  2652,  4533,  2138,  1045,  2572,
         1037,  4368,  2100,  2711,  2029,  2064,  2022,  2437,  2026,  2568,
         2514,  2104,  2202,  5165,  2006,  2033,  2029,  2064,  2022,  5094,
         2005,  1996,  3778,  8304,  1012,  2026,  7570, 27982,  2065,  2057,
         2024,  2652,  4533,  1010,  2026,  2568,  2003,  4208,  2006,  2055,
         1996,  2208,  1010,  2129,  2000,  2663,  2130,  2130,  3279,  2041,
        26176,  1996,  2208,  1012,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [131]:
test_audio_folder= "/kaggle/working/processed_audio/test"     # Audio_test embeddings.

audio_embeddings_test = []

for fname in test_df['filename']:
    audio_path = os.path.join(test_audio_folder, fname)
    emb = extract_audio_embedding(audio_path, wav2vec_model, wav2vec_processor, device)
    audio_embeddings_test.append(emb)


In [132]:
audio_embeddings_test = np.vstack(audio_embeddings_test)


audio_embeddings_test.shape

(204, 768)

In [133]:
X_test = np.concatenate([test_bert_embeds, audio_embeddings_test], axis=1)  #Test Concatenated embeddings




Y_pred= xgb_model.predict(X_test)



submission = pd.DataFrame({
    "filename":  test_df["filename"],  
    "label": Y_pred             
})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv saved with correct column order.")



         filename     label
0   audio_804.wav  3.183584
1  audio_1028.wav  2.911405
2   audio_865.wav  3.042996
3   audio_774.wav  3.511645
4  audio_1138.wav  3.030894
submission.csv saved with correct column order.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [07:42:29] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [07:42:29] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
